In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)
import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import linregress
from scipy.spatial.distance import cdist
import warnings

warnings.filterwarnings("ignore")


In [2]:
#Time period
OBSERVATION_END = pd.Timestamp("2017-06-30")
OBSERVATION_START = OBSERVATION_END - pd.DateOffset(months=24)  # July 2015
GAP_START = OBSERVATION_END + pd.DateOffset(months=1)  # July 2017
PREDICTION_START = pd.Timestamp("2017-08-01")
PREDICTION_END = pd.Timestamp("2018-01-31")

In [3]:
# this is the file path DO NOT CHANGE
# loyalty = pd.read_csv("Airline customer churn project/Customer Loyalty History.csv")
# flight = pd.read_csv("Airline customer churn project/Customer Flight Activity.csv")
# calendar = pd.read_csv("Airline customer churn project/Calendar.csv")

In [4]:
def load_data(history_path="Airline customer churn project/Customer Loyalty History.csv",
              activity_path="Airline customer churn project/Customer Flight Activity.csv"):
    """Load raw CSVs and return as DataFrames."""
    hist = pd.read_csv(history_path)
    flight = pd.read_csv(activity_path)
    return hist, flight

def clean_data(hist, flight):
    """
    Parse dates, handle missing values, anomalies.
    Returns cleaned DataFrames.
    """
    # ----- Parse enrollment & cancellation dates -----
    hist["Enrollment Date"] = pd.to_datetime(
        hist["Enrollment Year"].astype(str) + "-" +
        hist["Enrollment Month"].astype(str).str.zfill(2) + "-01",
        errors="coerce"
    )
    # Cancellation Year/Month are float due to NaN; convert valid entries
    mask_cancel = hist["Cancellation Year"].notna() & hist["Cancellation Month"].notna()
    hist.loc[mask_cancel, "Cancellation Date"] = pd.to_datetime(
        hist.loc[mask_cancel, "Cancellation Year"].astype(int).astype(str) + "-" +
        hist.loc[mask_cancel, "Cancellation Month"].astype(int).astype(str).str.zfill(2) + "-01",
        errors="coerce"
    )
    hist["Cancellation Date"] = pd.to_datetime(hist["Cancellation Date"], errors="coerce")

    # ----- Flight activity: combine Year/Month into a date -----
    flight["Activity Date"] = pd.to_datetime(
        flight["Year"].astype(str) + "-" +
        flight["Month"].astype(str).str.zfill(2) + "-01",
        errors="coerce"
    )

    # ----- Handle Salary missing values -----
    # Grouped median imputation by Education and Loyalty Card
    hist["Salary"] = hist.groupby(["Education", "Loyalty Card"])["Salary"].transform(
        lambda x: x.fillna(x.median())
    )
    # Fallback to global median if still missing (e.g., group entirely NaN)
    hist["Salary"] = hist["Salary"].fillna(hist["Salary"].median())

    # ----- Anomaly handling -----
    # Negative points: clip to 0 (or set to NaN and then 0)
    for col in ["Points Accumulated", "Points Redeemed", "Dollar Cost Points Redeemed"]:
        flight[col] = flight[col].clip(lower=0)

    # Rows where Total Flights > 0 but Distance == 0: flag and set Distance to NaN,
    # then impute with median distance per flight of that customer
    anomaly_mask = (flight["Total Flights"] > 0) & (flight["Distance"] == 0)
    if anomaly_mask.any():
        print(f"Warning: {anomaly_mask.sum()} rows with Flights>0 and Distance==0. Imputing Distance.")
    flight.loc[anomaly_mask, "Distance"] = np.nan
    # Impute using median distance per flight per customer (or global if needed)
    flight["Distance"] = flight.groupby("Loyalty Number")["Distance"].transform(
        lambda x: x.fillna(x.median())
    )
    flight["Distance"] = flight["Distance"].fillna(0)

    return hist, flight

In [5]:
# ==============================
# 2. FEATURE ENGINEERING (OBSERVATION WINDOW ONLY)
# ==============================
def compute_gini(array):
    """Gini coefficient of an array of values."""
    array = np.array(array, dtype=np.float64)
    if np.amin(array) < 0:
        array -= np.amin(array)  # make non-negative
    array = np.sort(array)
    n = array.shape[0]
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * array) - (n + 1) * np.sum(array)) / (n * np.sum(array) + 1e-10)

def build_monthly_time_series(customer_id, flight_df, start, end):
    """
    Create a complete monthly series for one customer between start and end.
    Returns a DataFrame with a 'Date' column (first of each month) and
    aggregated metrics (Total Flights, Points Accumulated, Points Redeemed).
    Missing months are filled with 0.
    """
    months = pd.date_range(start=start, end=end, freq="MS")
    df_cust = flight_df[(flight_df["Loyalty Number"] == customer_id) &
                        (flight_df["Activity Date"] >= start) &
                        (flight_df["Activity Date"] <= end)].copy()
    if df_cust.empty:
        # Return all-zero series
        return pd.DataFrame({
            "Date": months,
            "Total Flights": 0,
            "Points Accumulated": 0,
            "Points Redeemed": 0,
        })
    # Aggregate by month (just in case multiple entries per month)
    monthly = df_cust.groupby(pd.Grouper(key="Activity Date", freq="MS")).agg({
        "Total Flights": "sum",
        "Points Accumulated": "sum",
        "Points Redeemed": "sum",
    }).reset_index().rename(columns={"Activity Date": "Date"})
    # Merge with full month range
    full = pd.DataFrame({"Date": months})
    monthly = full.merge(monthly, on="Date", how="left").fillna(0)
    return monthly

def safe_linregress_slope(series):
    """Return slope of linear regression on a pandas Series; 0 if not possible."""
    if len(series) < 2 or series.std() == 0:
        return 0.0
    slope, _, _, _, _ = linregress(np.arange(len(series)), series.values)
    return slope

def engineer_features(customers, flight_df):
    """
    For each customer, extract features using only data up to OBSERVATION_END.
    Returns a DataFrame indexed by 'Loyalty Number' with engineered features.
    """
    features = []
    # Pre-compute loyalty card encoding
    card_order = {"Star": 0, "Nova": 1, "Aurora": 2}
    customers["Card Tier"] = customers["Loyalty Card"].map(card_order)

    # For Salary tier: compute quartiles once
    salary_quartiles = pd.qcut(customers["Salary"], q=4, labels=False, duplicates="drop")

    for _, row in customers.iterrows():
        lnum = row["Loyalty Number"]
        # ----- Monthly time series for observation window (24 months) -----
        ts = build_monthly_time_series(lnum, flight_df,
                                       OBSERVATION_START, OBSERVATION_END)
        # The series is 24 months long (or shorter if customer enrolled after start,
        # but we fill with zeros so it's always 24 rows).
        flights = ts["Total Flights"].values
        points_acc = ts["Points Accumulated"].values
        points_red = ts["Points Redeemed"].values
        months = ts["Date"]

        # ----- Flight Frequency Signals -----
        avg_full = flights.mean() if len(flights) > 0 else 0
        avg_last3 = flights[-3:].mean() if len(flights) >= 3 else avg_full
        flight_velocity_ratio = avg_last3 / avg_full if avg_full > 0 else 0.0

        # Flight trend slope over last 12 months
        if len(flights) >= 12:
            flight_trend_slope = safe_linregress_slope(pd.Series(flights[-12:]))
        else:
            flight_trend_slope = 0.0

        # Inter-flight gap acceleration (last 12 months)
        active_months = np.where(flights[-12:] > 0)[0]  # indices relative to -12
        if len(active_months) >= 2:
            gaps = np.diff(active_months)
            # slope of gaps over the sequence of active months
            gap_acceleration = safe_linregress_slope(pd.Series(gaps))
        else:
            gap_acceleration = 0.0

        # ----- Point Economy Signals -----
        # Redemption ratio: cumulative over entire history up to OBSERVATION_END
        history = flight_df[(flight_df["Loyalty Number"] == lnum) &
                            (flight_df["Activity Date"] <= OBSERVATION_END)]
        cum_acc = history["Points Accumulated"].sum()
        cum_red = history["Points Redeemed"].sum()
        redemption_ratio = cum_red / cum_acc if cum_acc > 0 else 0.0

        # Earn-burn velocity divergence (last 6 months)
        if len(flights) >= 6:
            slope_acc = safe_linregress_slope(pd.Series(points_acc[-6:]))
            slope_red = safe_linregress_slope(pd.Series(points_red[-6:]))
            earn_burn_divergence = slope_acc - slope_red
        else:
            earn_burn_divergence = 0.0

        # Redemption recency: months since last redemption
        redemptions = points_red > 0
        if redemptions.any():
            last_red_index = np.where(redemptions)[0][-1]
            # Count months from that month to end of observation window
            months_back = len(flights) - 1 - last_red_index
        else:
            months_back = 999  # large number for never redeemed
        redemption_recency = months_back

        # ----- Seasonal Concentration -----
        # Aggregate flights by quarter within observation window
        ts["Quarter"] = ts["Date"].dt.quarter
        quarter_flights = ts.groupby("Quarter")["Total Flights"].sum()
        # Ensure all 4 quarters present
        all_q = pd.Series([1,2,3,4], name="Quarter")
        quarter_flights = quarter_flights.reindex(all_q, fill_value=0)
        seasonal_gini = compute_gini(quarter_flights.values)

        # Peak quarter dominance: max over years
        ts["Year"] = ts["Date"].dt.year
        year_q = ts.groupby(["Year", "Quarter"])["Total Flights"].sum().reset_index()
        year_q["year_total"] = year_q.groupby("Year")["Total Flights"].transform("sum")
        year_q["dominance"] = year_q["Total Flights"] / year_q["year_total"].replace(0, 1e-10)
        peak_quarter_dominance = year_q["dominance"].max() if not year_q.empty else 0.0

        # ----- Demographic Interaction Terms -----
        # Salary tier (quartile based on whole dataset)
        sal_quartile = salary_quartiles.loc[row.name]  # row index matches customers index
        card_tier = row["Card Tier"]
        # mismatch: high salary (top quartile, 3) & low card (Star=0) OR low salary (0) & high card (Aurora=2)
        mismatch = 1 if ((sal_quartile == 3 and card_tier == 0) or
                         (sal_quartile == 0 and card_tier == 2)) else 0

        # Education encoding (ordinal)
        edu_order = {"High School": 0, "Bachelor": 1, "Master": 2, "Doctor": 3}
        edu_encoded = edu_order.get(row["Education"], 0)
        education_x_velocity = edu_encoded * flight_velocity_ratio

        # ----- Compile all features -----
        feat = {
            "Loyalty Number": lnum,
            "flight_velocity_ratio": flight_velocity_ratio,
            "flight_trend_slope": flight_trend_slope,
            "inter_flight_gap_acceleration": gap_acceleration,
            "redemption_ratio": redemption_ratio,
            "earn_burn_velocity_divergence": earn_burn_divergence,
            "redemption_recency": redemption_recency,
            "seasonal_gini": seasonal_gini,
            "peak_quarter_dominance": peak_quarter_dominance,
            "salary_tier_card_mismatch": mismatch,
            "education_x_velocity": education_x_velocity,
            # Additional static demographics/CLV for clustering
            "CLV": row["CLV"],
            "Salary": row["Salary"],
            "Card Tier": card_tier,
            "Enrollment Date": row["Enrollment Date"],
            "Cancellation Date": row["Cancellation Date"] if pd.notna(row["Cancellation Date"]) else pd.NaT,
        }
        features.append(feat)

    feature_df = pd.DataFrame(features).set_index("Loyalty Number")
    # Add tenure (months from enrollment to OBSERVATION_END)
    months_diff = (
        (OBSERVATION_END.year - feature_df["Enrollment Date"].dt.year) * 12
        + (OBSERVATION_END.month - feature_df["Enrollment Date"].dt.month)
    )
    feature_df["tenure_months"] = months_diff.clip(lower=0)
    # Points balance at observation end (cumulative up to that date)
    feature_df["points_balance"] = feature_df.apply(
        lambda r: flight_df[(flight_df["Loyalty Number"] == r.name) &
                            (flight_df["Activity Date"] <= OBSERVATION_END)][["Points Accumulated", "Points Redeemed"]]
                  .pipe(lambda g: g["Points Accumulated"].sum() - g["Points Redeemed"].sum()),
        axis=1
    )
    return feature_df


In [6]:
# hist, flight = load_data()
# hist, flight = clean_data(hist, flight)
# features = engineer_features(hist, flight)
# target = label_churn(hist, flight)
# data = features.join(target, how="inner")
# target.value_counts(normalize=True)

In [7]:
import pandas as pd
import numpy as np

# ======================
# 0. GLOBAL CONSTANTS
# ======================
OBSERVATION_START = pd.Timestamp("2015-07-01")
OBSERVATION_END   = pd.Timestamp("2017-06-30")
PREDICTION_START  = pd.Timestamp("2017-08-01")
PREDICTION_END    = pd.Timestamp("2018-01-31")

# ======================
# 1. LOAD & LIGHT CLEAN
# ======================
hist = pd.read_csv("Airline customer churn project/Customer Loyalty History.csv")
flight = pd.read_csv("Airline customer churn project/Customer Flight Activity.csv")

# Parse Activity Date
flight["Activity Date"] = pd.to_datetime(
    flight["Year"].astype(str) + "-" + flight["Month"].astype(str).str.zfill(2) + "-01"
)

# (Optional: clip negative points, not needed for churn conditions)
flight["Points Accumulated"] = flight["Points Accumulated"].clip(lower=0)
flight["Points Redeemed"]   = flight["Points Redeemed"].clip(lower=0)

# ======================
# 2. BUILD COMPLETE MONTHLY GRID FOR ALL CUSTOMERS
# ======================
# All months from observation start to prediction end
all_months = pd.date_range(OBSERVATION_START, PREDICTION_END, freq="MS")
cust_ids = flight["Loyalty Number"].unique()

# Create full grid (Loyalty Number, Month)
grid = pd.MultiIndex.from_product([cust_ids, all_months], names=["Loyalty Number", "Month"])
grid_df = pd.DataFrame(index=grid).reset_index()

# Aggregate flight data to monthly totals per customer
monthly_agg = flight.groupby(["Loyalty Number", pd.Grouper(key="Activity Date", freq="MS")]) \
                    .agg({"Total Flights": "sum", "Points Accumulated": "sum", "Points Redeemed": "sum"}) \
                    .reset_index().rename(columns={"Activity Date": "Month"})

# Left join to grid → missing months become NaN → fill with 0
monthly = grid_df.merge(monthly_agg, on=["Loyalty Number", "Month"], how="left").fillna(0)
monthly["Month"] = pd.to_datetime(monthly["Month"])  # ensure datetime

# ======================
# 3. COMPUTE CHURN CONDITIONS
# ======================

def label_churn(customers, flight_df):
    """
    Compute binary churn label based on the prediction window Aug 2017 - Jan 2018.
    Condition 3 tightened: balance > 15,000, >0 lifetime flights before the 18-month
    stagnation window, and exactly 0 flights & redemptions in the last 18 months.
    """
    # 18-month stagnation window: Aug 2016 – Jan 2018
    STAGNATION_START = PREDICTION_END - pd.DateOffset(months=17)  # Aug 2016
    STAGNATION_END   = PREDICTION_END                               # Jan 2018

    labels = {}
    for lnum in customers["Loyalty Number"]:
        # --- Condition 1: Total Ghosting in prediction window ---
        pred_mask = (flight_df["Loyalty Number"] == lnum) & \
                    (flight_df["Activity Date"] >= PREDICTION_START) & \
                    (flight_df["Activity Date"] <= PREDICTION_END)
        pred_df = flight_df[pred_mask]
        total_flights_pred = pred_df["Total Flights"].sum()
        total_acc_pred = pred_df["Points Accumulated"].sum()
        total_red_pred = pred_df["Points Redeemed"].sum()
        ghost = (total_flights_pred == 0) and (total_acc_pred == 0) and (total_red_pred == 0)

        # --- Condition 2: Behavioral drop-off ---
        ts_pred = build_monthly_time_series(lnum, flight_df, PREDICTION_START, PREDICTION_END)
        flights_pred = ts_pred["Total Flights"].values
        ts_obs = build_monthly_time_series(lnum, flight_df, OBSERVATION_START, OBSERVATION_END)
        baseline = ts_obs["Total Flights"].mean()
        dropoff = False
        if len(flights_pred) >= 3:
            rolled = pd.Series(flights_pred).rolling(window=3, min_periods=1).mean().values
            below = rolled < (0.3 * baseline)
            for i in range(len(below) - 1):
                if below[i] and below[i + 1]:
                    dropoff = True
                    break

        # --- Condition 3: Credit-Card-Only Stagnation (tightened) ---
        # Build monthly series for the stagnation window only
        ts_stag = build_monthly_time_series(lnum, flight_df, STAGNATION_START, STAGNATION_END)
        # Strictly 0 flights and 0 redemptions during the 18 months
        stag_zero = (ts_stag["Total Flights"].sum() == 0) and (ts_stag["Points Redeemed"].sum() == 0)

        # Cumulative lifetime flights before the stagnation window
        before_stag = flight_df[(flight_df["Loyalty Number"] == lnum) &
                                (flight_df["Activity Date"] < STAGNATION_START)]
        lifetime_flights_before = before_stag["Total Flights"].sum()

        # Running points balance at end of prediction window
        full_hist = flight_df[(flight_df["Loyalty Number"] == lnum) &
                              (flight_df["Activity Date"] <= PREDICTION_END)]
        balance = full_hist["Points Accumulated"].sum() - full_hist["Points Redeemed"].sum()

        stagnation = (
            stag_zero and
            (lifetime_flights_before > 0) and
            (balance > 15000)
        )

        churn = ghost or dropoff or stagnation
        labels[lnum] = int(churn)

    return pd.Series(labels, name="is_churned")

# ======================
# 4. COMBINE CONDITIONS
# ======================
# Align all indices to cust_ids
ghost = ghost.reindex(cust_ids, fill_value=False)
dropoff = dropoff.reindex(cust_ids, fill_value=False)
stagnation = stagnation.reindex(cust_ids, fill_value=False)

churn = ghost | dropoff | stagnation
churn_rate = churn.mean()
print(f"Baseline churn rate: {churn_rate:.2%}")
print("Condition 1 (Ghosting) rate:", ghost.mean() * 100)
print("Condition 2 (Drop-off) rate:", dropoff.mean() * 100)
print("Condition 3 (CC Stagnation) rate:", stagnation.mean() * 100)

# ======================
# 5. OPTIONAL: SAVE & TIME
# ======================
import time
start = time.time()
# ... (the above code) ...
print(f"Computed in {time.time()-start:.1f} seconds")

NameError: name 'ghost' is not defined

In [8]:
import pandas as pd
import numpy as np

# ==============================
# 0. TIME WINDOW CONSTANTS
# ==============================
OBSERVATION_START = pd.Timestamp("2015-07-01")
OBSERVATION_END   = pd.Timestamp("2017-06-30")
GAP_START         = pd.Timestamp("2017-07-01")   # not used directly, kept for completeness
PREDICTION_START  = pd.Timestamp("2017-08-01")
PREDICTION_END    = pd.Timestamp("2018-01-31")

# ==============================
# 1. HELPER FUNCTION
# ==============================
def build_monthly_time_series(customer_id, flight_df, start, end):
    """
    Create a complete monthly series for one customer between start and end.
    Returns a DataFrame with 'Date', 'Total Flights', 'Points Accumulated', 'Points Redeemed'.
    Missing months are filled with 0.
    """
    months = pd.date_range(start=start, end=end, freq="MS")
    df_cust = flight_df[(flight_df["Loyalty Number"] == customer_id) &
                        (flight_df["Activity Date"] >= start) &
                        (flight_df["Activity Date"] <= end)].copy()
    if df_cust.empty:
        return pd.DataFrame({
            "Date": months,
            "Total Flights": 0,
            "Points Accumulated": 0,
            "Points Redeemed": 0,
        })
    # Aggregate by month (in case of multiple records)
    monthly = df_cust.groupby(pd.Grouper(key="Activity Date", freq="MS")).agg({
        "Total Flights": "sum",
        "Points Accumulated": "sum",
        "Points Redeemed": "sum",
    }).reset_index().rename(columns={"Activity Date": "Date"})
    # Merge with full month range, fill missing with 0
    full = pd.DataFrame({"Date": months})
    monthly = full.merge(monthly, on="Date", how="left").fillna(0)
    return monthly

# ==============================
# 2. CHURN LABELING (TIGHTENED CONDITION 3)
# ==============================
def label_churn(customers, flight_df):
    """
    Compute binary churn label based on the prediction window Aug 2017 - Jan 2018.
    Condition 3 tightened: balance > 15,000, >0 lifetime flights before the 18-month
    stagnation window, and exactly 0 flights & 0 redemptions during that window.
    """
    # Stagnation window: 18 months ending at PREDICTION_END (Jan 2018) → starts Aug 2016
    STAGNATION_START = PREDICTION_END - pd.DateOffset(months=17)  # Aug 2016
    STAGNATION_END   = PREDICTION_END                               # Jan 2018

    labels = {}
    for lnum in customers["Loyalty Number"]:
        # ----------------------------------------------------------
        # Condition 1: Total Ghosting in prediction window
        # ----------------------------------------------------------
        pred_mask = (flight_df["Loyalty Number"] == lnum) & \
                    (flight_df["Activity Date"] >= PREDICTION_START) & \
                    (flight_df["Activity Date"] <= PREDICTION_END)
        pred_df = flight_df[pred_mask]
        total_flights_pred = pred_df["Total Flights"].sum()
        total_acc_pred = pred_df["Points Accumulated"].sum()
        total_red_pred = pred_df["Points Redeemed"].sum()
        ghost = (total_flights_pred == 0) and (total_acc_pred == 0) and (total_red_pred == 0)

        # ----------------------------------------------------------
        # Condition 2: Behavioral Drop-off
        # ----------------------------------------------------------
        ts_pred = build_monthly_time_series(lnum, flight_df, PREDICTION_START, PREDICTION_END)
        flights_pred = ts_pred["Total Flights"].values
        ts_obs = build_monthly_time_series(lnum, flight_df, OBSERVATION_START, OBSERVATION_END)
        baseline = ts_obs["Total Flights"].mean()
        dropoff = False
        if len(flights_pred) >= 3:
            rolled = pd.Series(flights_pred).rolling(window=3, min_periods=1).mean().values
            below = rolled < (0.3 * baseline)
            for i in range(len(below) - 1):
                if below[i] and below[i + 1]:
                    dropoff = True
                    break

        # ----------------------------------------------------------
        # Condition 3: Credit-Card-Only Stagnation (TIGHTENED)
        # ----------------------------------------------------------
        # 3a. Stagnation window: exactly 0 flights & 0 redemptions
        ts_stag = build_monthly_time_series(lnum, flight_df, STAGNATION_START, STAGNATION_END)
        stag_zero = (ts_stag["Total Flights"].sum() == 0) and (ts_stag["Points Redeemed"].sum() == 0)

        # 3b. Must have flown at least once BEFORE the stagnation window
        before_stag = flight_df[(flight_df["Loyalty Number"] == lnum) &
                                (flight_df["Activity Date"] < STAGNATION_START)]
        lifetime_flights_before = before_stag["Total Flights"].sum()

        # 3c. Running points balance at the end of the prediction window > 15,000
        full_hist = flight_df[(flight_df["Loyalty Number"] == lnum) &
                              (flight_df["Activity Date"] <= PREDICTION_END)]
        balance = full_hist["Points Accumulated"].sum() - full_hist["Points Redeemed"].sum()

        stagnation = (
            stag_zero and
            (lifetime_flights_before > 0) and
            (balance > 15000)
        )

        # Combine all three conditions
        churn = ghost or dropoff or stagnation
        labels[lnum] = int(churn)

    return pd.Series(labels, name="is_churned")
target = label_churn(hist, flight)
print("Churn rate:", target.mean())
print(target.value_counts(normalize=True))

Churn rate: 0.39290195375515324
is_churned
0    0.607098
1    0.392902
Name: proportion, dtype: float64
